# Supply Chain EDA — Exploratory Data Analysis
**Data:** suppliers · inventory · purchase_orders · stock_movements

## Business Questions
1. **Supplier Performance** — NCC nào quan trọng nhất? Ai hay bị cancelled?
2. **Inventory Health** — Tồn kho đang ở trạng thái nào? Category nào rủi ro?
3. **Procurement Analysis** — Xu hướng mua hàng? Lead time?
4. **Stock Movement Patterns** — Biến động kho theo tháng/category?


---
## Phần 0 — Setup & Data Preparation

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.figsize':    (12, 5),
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'font.size':         11,
    'axes.titlesize':    13,
    'axes.titleweight':  'bold',
})
COLORS = ['#2196F3','#4CAF50','#FF9800','#E91E63','#9C27B0']
sns.set_palette(COLORS)
print("Setup xong")


Setup xong


In [ ]:
import os

# ── ĐỌC DATA CLEAN ────────────────────────────────────────────────
# Ưu tiên đọc từ data/clean/ (đã qua pipeline cleaning)
# Nếu chưa có → báo lỗi rõ ràng, không clean lại từ raw
CLEAN_DIR = "../../data/clean/project-04-supply-chain"

required_files = {
    "suppliers":       f"{CLEAN_DIR}/suppliers_clean.csv",
    "inventory":       f"{CLEAN_DIR}/inventory_clean.csv",
    "purchase_orders": f"{CLEAN_DIR}/purchase_orders_clean.csv",
    "stock_movements": f"{CLEAN_DIR}/stock_movements_clean.csv",
}

# Kiểm tra file tồn tại
missing = [name for name, path in required_files.items() if not os.path.exists(path)]
if missing:
    print("FILE CLEAN CHUA CO:")
    for name in missing:
        print(f"  - {required_files[name]}")
    print()
    print("Cach tao file clean:")
    print("  Option A: Chay 04_supply_chain_cleaning.ipynb truoc")
    print("  Option B: Export CSV tu Google Sheets vao data/clean/project-04-supply-chain/")
    raise FileNotFoundError("Vui long tao file clean truoc khi chay EDA")

# Đọc data đã clean — không cần clean lại
sup = pd.read_csv(required_files["suppliers"])
inv = pd.read_csv(required_files["inventory"])
po  = pd.read_csv(required_files["purchase_orders"])
mov = pd.read_csv(required_files["stock_movements"])

# Chỉ parse kiểu dữ liệu — KHÔNG clean lại logic
po["order_date"]    = pd.to_datetime(po["order_date"],    errors="coerce")
po["expected_date"] = pd.to_datetime(po["expected_date"], errors="coerce")
po["lead_time_days"]= (po["expected_date"] - po["order_date"]).dt.days
po["order_ym"]      = po["order_date"].dt.strftime("%Y-%m")

mov["movement_date"] = pd.to_datetime(mov["movement_date"], errors="coerce")
mov["movement_ym"]   = mov["movement_date"].dt.strftime("%Y-%m")
mov["quantity"]      = pd.to_numeric(mov["quantity"], errors="coerce").fillna(0)

inv["inventory_value"] = pd.to_numeric(inv["stock_qty"], errors="coerce").fillna(0) *                          pd.to_numeric(inv["unit_cost_vnd"], errors="coerce").fillna(0)
# Map needs_reorder — xử lý cả có dấu (GAS output) lẫn không dấu (Python output)
# GAS xuất ra: "Có" / "Không"
# Python xuất ra: "Có" / "Không" (hoặc True/False tùy version)
inv["needs_reorder"] = inv["needs_reorder"].astype(str).str.strip().map({
    "Có": True,   "Khong": False,   # có dấu (GAS)
    "Co": True,   "Không": False,   # không dấu
    "True": True, "False": False,   # boolean string
    "1": True,    "0": False,       # số
}).fillna(False)

# Join để phân tích cross-table
mov     = mov.merge(inv[["item_id","category"]], on="item_id", how="left")
po_full = po.merge(sup[["supplier_id","supplier_name","city","payment_terms"]], on="supplier_id", how="left")
po_full = po_full.merge(inv[["item_id","category"]], on="item_id", how="left")

print("Data clean loaded:")
print(f"  Suppliers: {len(sup)} | Inventory: {len(inv)} | PO: {len(po)} | Movements: {len(mov)}")
print(f"  Tong gia tri PO: {po['total_amount'].sum()/1e9:.2f} ty VND")
print(f"  PO date range: {po['order_date'].min().date()} -> {po['order_date'].max().date()}")

---
## Nhom 1 — Supplier Performance

In [ ]:
# 1A — Top 10 NCC theo tong gia tri PO
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

top_sup = (po_full.groupby("supplier_name")["total_amount"]
           .sum().sort_values(ascending=False).head(10))
axes[0].barh(range(len(top_sup)), top_sup.values / 1e6,
             color=COLORS[0], alpha=0.85)
axes[0].set_yticks(range(len(top_sup)))
axes[0].set_yticklabels([s[:25] for s in top_sup.index], fontsize=9)
axes[0].set_xlabel("Tong gia tri PO (trieu VND)")
axes[0].set_title("Top 10 NCC theo gia tri PO")
for i, v in enumerate(top_sup.values / 1e6):
    axes[0].text(v + 2, i, f"{v:.0f}M", va="center", fontsize=8)

# Ti le cancel theo NCC
status_by_sup = (po_full[po_full["supplier_name"].isin(top_sup.index)]
                 .groupby(["supplier_name","status"]).size().unstack(fill_value=0))
if "cancelled" in status_by_sup.columns:
    status_by_sup["cancel_rate"] = status_by_sup["cancelled"] / status_by_sup.sum(axis=1) * 100
    cr = status_by_sup["cancel_rate"].sort_values(ascending=False)
    bar_colors = [COLORS[3] if v > 30 else COLORS[0] for v in cr.values]
    axes[1].bar(range(len(cr)), cr.values, color=bar_colors, alpha=0.85)
    axes[1].set_xticks(range(len(cr)))
    axes[1].set_xticklabels([s[:18] for s in cr.index], rotation=45, ha="right", fontsize=8)
    axes[1].axhline(30, color="red", linestyle="--", alpha=0.6, label="Nguong 30%")
    axes[1].set_ylabel("Ti le cancelled (%)")
    axes[1].set_title("Ti le cancelled PO theo NCC (do = > 30%)")
    axes[1].legend()

plt.tight_layout()
plt.savefig("01_supplier_performance.png", dpi=120, bbox_inches="tight")
plt.show()

top3_pct = top_sup.values[:3].sum() / po["total_amount"].sum() * 100
cancel_pct = (po["status"]=="cancelled").mean() * 100
print(f"INSIGHT: Top 3 NCC chiem {top3_pct:.1f}% tong gia tri PO")
print(f"INSIGHT: Ti le cancelled PO toan he thong: {cancel_pct:.1f}%")


In [ ]:
# 1B — Payment terms & NCC theo thanh pho
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

pt_counts = sup["payment_terms"].value_counts()
axes[0].pie(pt_counts.values, labels=pt_counts.index,
            colors=COLORS[:len(pt_counts)], autopct="%1.1f%%", startangle=90,
            wedgeprops={"edgecolor":"white","linewidth":1.5})
axes[0].set_title("Phan bo Payment Terms")

city_val = po_full.groupby("city")["total_amount"].sum().sort_values(ascending=False)
axes[1].bar(city_val.index, city_val.values / 1e6, color=COLORS[2], alpha=0.85)
axes[1].set_ylabel("Tong gia tri PO (trieu VND)")
axes[1].set_title("Gia tri PO theo thanh pho NCC")
for i, (city, val) in enumerate(zip(city_val.index, city_val.values)):
    axes[1].text(i, val/1e6 + 5, f"{val/1e6:.0f}M", ha="center", fontsize=9)

plt.tight_layout()
plt.savefig("01_payment_city.png", dpi=120, bbox_inches="tight")
plt.show()


---
## Nhom 2 — Inventory Health

In [ ]:
# 2A — Gia tri ton kho va tinh trang theo category
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

inv_cat = (inv.groupby("category")
           .agg(total_value=("inventory_value","sum"),
                item_count=("item_id","count"),
                needs_reorder=("needs_reorder","sum"))
           .sort_values("total_value", ascending=False))

bars = axes[0].bar(inv_cat.index, inv_cat["total_value"]/1e6,
                   color=COLORS[:len(inv_cat)], alpha=0.85)
axes[0].set_ylabel("Gia tri ton kho (trieu VND)")
axes[0].set_title("Gia tri ton kho theo category")
for bar, val in zip(bars, inv_cat["total_value"].values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f"{val/1e6:.0f}M", ha="center", fontsize=9)

normal  = inv_cat["item_count"] - inv_cat["needs_reorder"]
reorder = inv_cat["needs_reorder"]
x = range(len(inv_cat))
axes[1].bar(x, normal.values, label="Binh thuong", color=COLORS[1], alpha=0.85)
axes[1].bar(x, reorder.values, bottom=normal.values,
            label="Can dat hang", color=COLORS[3], alpha=0.85)
axes[1].set_xticks(x)
axes[1].set_xticklabels(inv_cat.index)
axes[1].set_ylabel("So luong items")
axes[1].set_title("Tinh trang ton kho theo category")
axes[1].legend()

plt.tight_layout()
plt.savefig("02_inventory_health.png", dpi=120, bbox_inches="tight")
plt.show()

print(f"INSIGHT: {inv['needs_reorder'].sum()}/{len(inv)} items can dat hang ngay ({inv['needs_reorder'].mean()*100:.1f}%)")
print(f"INSIGHT: Tong gia tri ton kho: {inv['inventory_value'].sum()/1e6:.0f} trieu VND")


In [ ]:
# 2B — Top 15 items gia tri ton kho cao nhat
fig, ax = plt.subplots(figsize=(13, 6))

top_items = inv.nlargest(15, "inventory_value")[
    ["item_id","item_name","category","stock_qty","unit_cost_vnd","inventory_value","needs_reorder"]]
colors_items = [COLORS[3] if r else COLORS[0] for r in top_items["needs_reorder"]]
ax.barh(range(len(top_items)), top_items["inventory_value"].values / 1e6,
        color=colors_items, alpha=0.85)
ax.set_yticks(range(len(top_items)))
ax.set_yticklabels([f"{r['item_id']} - {r['item_name'][:20]}"
                    for _, r in top_items.iterrows()], fontsize=9)
ax.set_xlabel("Gia tri ton kho (trieu VND)")
ax.set_title("Top 15 Items gia tri ton kho cao nhat (do = can dat hang ngay)")
for i, val in enumerate(top_items["inventory_value"].values / 1e6):
    ax.text(val + 0.3, i, f"{val:.0f}M", va="center", fontsize=8)
plt.tight_layout()
plt.savefig("02_top_items.png", dpi=120, bbox_inches="tight")
plt.show()


---
## Nhom 3 — Procurement Analysis

In [ ]:
# 3A — Xu huong gia tri PO theo thang
fig, axes = plt.subplots(2, 1, figsize=(14, 9))

monthly = (po_full[po_full["status"]!="cancelled"]
           .groupby("order_ym")["total_amount"].sum().reset_index())
monthly["val_m"] = monthly["total_amount"] / 1e6

x = range(len(monthly))
axes[0].fill_between(x, monthly["val_m"], alpha=0.25, color=COLORS[0])
axes[0].plot(x, monthly["val_m"], color=COLORS[0], linewidth=2, marker="o", markersize=4)
axes[0].set_xticks(range(0, len(monthly), 2))
axes[0].set_xticklabels(monthly["order_ym"].iloc[::2], rotation=45, ha="right", fontsize=8)
axes[0].set_ylabel("Gia tri PO (trieu VND)")
axes[0].set_title("Xu huong gia tri PO theo thang (exclude cancelled)")
avg = monthly["val_m"].mean()
axes[0].axhline(avg, color="red", linestyle="--", alpha=0.7, label=f"TB: {avg:.0f}M")
axes[0].legend()

cat_monthly = (po_full[po_full["status"]!="cancelled"]
               .groupby(["order_ym","category"])["total_amount"]
               .sum().unstack(fill_value=0) / 1e6)
cat_monthly.plot(kind="bar", stacked=True, ax=axes[1],
                 color=COLORS[:len(cat_monthly.columns)], alpha=0.85)
axes[1].set_xticklabels(cat_monthly.index, rotation=45, ha="right", fontsize=8)
axes[1].set_ylabel("Gia tri PO (trieu VND)")
axes[1].set_title("Gia tri PO theo thang va category")
axes[1].legend(loc="upper right", fontsize=8)

plt.tight_layout()
plt.savefig("03_procurement_trend.png", dpi=120, bbox_inches="tight")
plt.show()


In [ ]:
# 3B — Lead time analysis
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

valid_lt = po_full[(po_full["lead_time_days"] > 0) &
                   (po_full["lead_time_days"] < 60) &
                   (po_full["status"] != "cancelled")]["lead_time_days"]

axes[0].hist(valid_lt, bins=20, color=COLORS[0], alpha=0.8, edgecolor="white")
axes[0].axvline(valid_lt.mean(),   color="red",    linestyle="--", label=f"TB: {valid_lt.mean():.1f} ngay")
axes[0].axvline(valid_lt.median(), color="orange", linestyle="--", label=f"Median: {valid_lt.median():.1f} ngay")
axes[0].set_xlabel("Lead time (ngay)")
axes[0].set_ylabel("So luong PO")
axes[0].set_title("Phan bo Lead Time")
axes[0].legend()

lt_cat = (po_full[(po_full["lead_time_days"]>0) & (po_full["lead_time_days"]<60)]
          .groupby("category")["lead_time_days"]
          .agg(["mean","std"]).round(1).sort_values("mean", ascending=False))
axes[1].barh(range(len(lt_cat)), lt_cat["mean"], xerr=lt_cat["std"],
             color=COLORS[2], alpha=0.85, error_kw={"capsize":4})
axes[1].set_yticks(range(len(lt_cat)))
axes[1].set_yticklabels(lt_cat.index)
axes[1].set_xlabel("Lead time trung binh (ngay)")
axes[1].set_title("Lead time trung binh theo category")

plt.tight_layout()
plt.savefig("03_lead_time.png", dpi=120, bbox_inches="tight")
plt.show()

print(f"INSIGHT: Lead time TB = {valid_lt.mean():.1f} ngay | Median = {valid_lt.median():.1f} ngay")
print(f"INSIGHT: {(valid_lt>21).mean()*100:.1f}% PO co lead time > 21 ngay")


---
## Nhom 4 — Stock Movement Patterns

In [ ]:
# 4A — Phan bo movement types
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

type_qty = mov.groupby("movement_type")["quantity"].sum().sort_values(ascending=False)
axes[0].bar(type_qty.index, type_qty.values, color=COLORS[:len(type_qty)], alpha=0.85)
axes[0].set_ylabel("Tong so luong")
axes[0].set_title("Tong so luong theo movement type")
for i, (t, v) in enumerate(zip(type_qty.index, type_qty.values)):
    axes[0].text(i, v + 20, f"{v:,.0f}", ha="center", fontsize=9)

type_cat = mov.groupby(["movement_type","category"]).size().unstack(fill_value=0)
type_cat.plot(kind="bar", ax=axes[1], color=COLORS[:len(type_cat.columns)], alpha=0.85)
axes[1].set_xticklabels(type_cat.index, rotation=0)
axes[1].set_ylabel("So lan movement")
axes[1].set_title("So lan movement theo type va category")
axes[1].legend(loc="upper right", fontsize=8)

plt.tight_layout()
plt.savefig("04_movement_types.png", dpi=120, bbox_inches="tight")
plt.show()


In [ ]:
# 4B — IN vs OUT theo thang + Net flow
fig, axes = plt.subplots(2, 1, figsize=(14, 9))

in_out = (mov[mov["movement_type"].isin(["IN","OUT"])]
          .groupby(["movement_ym","movement_type"])["quantity"]
          .sum().unstack(fill_value=0))

x = range(len(in_out))
w = 0.35
in_vals  = in_out.get("IN",  pd.Series(0, index=in_out.index)).values
out_vals = in_out.get("OUT", pd.Series(0, index=in_out.index)).values

axes[0].bar([i-w/2 for i in x], in_vals,  width=w, label="IN",  color=COLORS[1], alpha=0.85)
axes[0].bar([i+w/2 for i in x], out_vals, width=w, label="OUT", color=COLORS[3], alpha=0.85)
axes[0].set_xticks(x)
axes[0].set_xticklabels(in_out.index, rotation=45, ha="right", fontsize=8)
axes[0].set_ylabel("Tong so luong")
axes[0].set_title("Bien dong nhap/xuat kho theo thang")
axes[0].legend()

net = in_vals - out_vals
bar_c = [COLORS[1] if v >= 0 else COLORS[3] for v in net]
axes[1].bar(x, net, color=bar_c, alpha=0.85)
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].set_xticks(x)
axes[1].set_xticklabels(in_out.index, rotation=45, ha="right", fontsize=8)
axes[1].set_ylabel("Net flow (IN - OUT)")
axes[1].set_title("Net flow ton kho theo thang (xanh=tang, do=giam)")

plt.tight_layout()
plt.savefig("04_movement_trend.png", dpi=120, bbox_inches="tight")
plt.show()


In [ ]:
# 4C — Top items vong quay cao + ti le return
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

top_out = (mov[mov["movement_type"]=="OUT"]
           .groupby("item_id")["quantity"].sum()
           .sort_values(ascending=False).head(15))
top_out_df = top_out.reset_index().merge(
    inv[["item_id","item_name","category"]], on="item_id")
cat_colors = {cat: COLORS[i] for i, cat in enumerate(inv["category"].unique())}
c_out = [cat_colors.get(c, "#999") for c in top_out_df["category"]]

axes[0].barh(range(len(top_out_df)), top_out_df["quantity"].values, color=c_out, alpha=0.85)
axes[0].set_yticks(range(len(top_out_df)))
axes[0].set_yticklabels([f"{r['item_id']} ({r['category'][:5]})" for _,r in top_out_df.iterrows()], fontsize=9)
axes[0].set_xlabel("Tong so luong xuat kho")
axes[0].set_title("Top 15 Items xuat kho nhieu nhat")

ret_cat = (mov[mov["movement_type"].isin(["IN","RETURN"])]
           .groupby(["category","movement_type"])["quantity"]
           .sum().unstack(fill_value=0))
if "RETURN" in ret_cat.columns and "IN" in ret_cat.columns:
    ret_cat["return_rate"] = (ret_cat["RETURN"] / ret_cat["IN"] * 100).round(1)
    axes[1].bar(ret_cat.index, ret_cat["return_rate"],
                color=[cat_colors.get(c,"#999") for c in ret_cat.index], alpha=0.85)
    axes[1].set_ylabel("Ti le return / nhap (%)")
    axes[1].set_title("Ti le tra hang theo category")
    for i, (cat, row) in enumerate(ret_cat.iterrows()):
        axes[1].text(i, row["return_rate"] + 0.1, f"{row['return_rate']:.1f}%", ha="center", fontsize=9)

plt.tight_layout()
plt.savefig("04_stock_turnover.png", dpi=120, bbox_inches="tight")
plt.show()

print(f"INSIGHT: Item xuat nhieu nhat: {top_out_df.iloc[0]['item_id']} - {top_out_df.iloc[0]['item_name']}")


---
## Phan 5 — Tong hop Insights & Recommendations

In [ ]:
print("=" * 65)
print("SUPPLY CHAIN EDA - TOM TAT INSIGHTS")
print("=" * 65)

top3_sup = po_full.groupby("supplier_name")["total_amount"].sum().nlargest(3)
top3_pct = top3_sup.sum() / po["total_amount"].sum() * 100
cancel_pct = (po["status"]=="cancelled").mean() * 100
reorder_cnt = inv["needs_reorder"].sum()
total_inv = inv["inventory_value"].sum()
valid_lt = po_full[(po_full["lead_time_days"]>0) & (po_full["lead_time_days"]<60)]["lead_time_days"]
in_qty  = mov[mov["movement_type"]=="IN"]["quantity"].sum()
out_qty = mov[mov["movement_type"]=="OUT"]["quantity"].sum()
ret_qty = mov[mov["movement_type"]=="RETURN"]["quantity"].sum()

print()
print("[SUPPLIER PERFORMANCE]")
print(f"  - Top 3 NCC chiem {top3_pct:.1f}% tong gia tri PO")
print(f"  - Ti le cancelled PO: {cancel_pct:.1f}%")
print(f"  - Payment terms: {sup['payment_terms'].value_counts().to_dict()}")

print()
print("[INVENTORY HEALTH]")
print(f"  - {reorder_cnt}/{len(inv)} items ({reorder_cnt/len(inv)*100:.1f}%) can dat hang ngay")
print(f"  - Tong gia tri ton kho: {total_inv/1e6:.0f} trieu VND")
top_cat = inv.groupby("category")["inventory_value"].sum().idxmax()
print(f"  - Category gia tri cao nhat: {top_cat}")

print()
print("[PROCUREMENT]")
monthly_avg = po_full[po_full["status"]!="cancelled"].groupby("order_ym")["total_amount"].sum().mean()
print(f"  - Gia tri PO TB/thang: {monthly_avg/1e6:.0f} trieu VND")
print(f"  - Lead time TB: {valid_lt.mean():.1f} ngay | Median: {valid_lt.median():.1f} ngay")
print(f"  - {(valid_lt>21).mean()*100:.1f}% PO co lead time > 21 ngay")

print()
print("[STOCK MOVEMENTS]")
print(f"  - Tong nhap: {in_qty:,.0f} | Tong xuat: {out_qty:,.0f} units")
print(f"  - Net flow: {in_qty-out_qty:+,.0f} units")
print(f"  - Ti le return: {ret_qty/in_qty*100:.1f}% so voi nhap kho")

print()
print("=" * 65)
print("RECOMMENDATIONS")
print("=" * 65)
top_reorder_cat = inv[inv["needs_reorder"]]["category"].value_counts().idxmax()
top_lt_cat = po_full[po_full["lead_time_days"]>21]["category"].value_counts().idxmax() if len(po_full[po_full["lead_time_days"]>21]) > 0 else "N/A"
print(f"1. DAT HANG NGAY: {reorder_cnt} items xuong duoi reorder point")
print(f"   - Uu tien category: {top_reorder_cat}")
print(f"2. REVIEW CANCELLED PO ({cancel_pct:.1f}% - cao hon benchmark 10-15%)")
print(f"   - Hop voi procurement team de xac dinh nguyen nhan")
print(f"3. LEAD TIME DAI (>21 ngay): xem xet tang safety stock")
print(f"   - Category rui ro nhat: {top_lt_cat}")
print(f"4. SUPPLIER CONCENTRATION: Top 3 NCC chiem {top3_pct:.1f}%")
print(f"   - De xuat da dang hoa nha cung cap de giam rui ro")
